# 5.1.6 Model 6: K-Nearest Neighbors (KNN) Regressor

Trained on `X_train_scaled.csv` (log1p-transformed numeric features, per Section 3.11.2) since KNN is distance-based and therefore scale-sensitive, same reasoning as Linear Regression, Ridge, and SVR.

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train_scaled.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test_scaled.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


X_train: (3004, 51) | X_test: (751, 51)


## Additional Feature Standardisation

Section 3.11 applies log1p to reduce skewness within individual features, but this does not equalise the *scale* of different features relative to each other - after log1p, Total Units still has a wider spread (std approx 0.69) than Bathroom (std approx 0.18), so a raw feature difference on Total Units would dominate a raw feature difference on Bathroom in any distance calculation, regardless of which is actually more predictive of price. Since KNN selects neighbours purely by distance, an additional StandardScaler step is applied here on top of the log1p-scaled data, fitted on X_train only and applied to both X_train and X_test (same leakage-avoidance principle as Section 3.6), so every feature contributes to the distance calculation on equal footing.

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Standardised - std should now be ~1.0 for every feature:")
print(X_train[["Property Size", "Bathroom", "Total Units"]].std())


Standardised - std should now be ~1.0 for every feature:
Property Size    1.000166
Bathroom         1.000166
Total Units      1.000166
dtype: float64


## Hyperparameter Tuning

The model was tuned using `GridSearchCV` with 5-fold cross-validation on `X_train`. Since `y_train` is `log(price)`, a plain RMSE scorer would rank configurations by log-scale error - because `exp()` is a non-linear transform, this does not guarantee the same ranking as RM-scale RMSE, which Section 1.8 defines as this project's primary metric. `GridSearchCV` is therefore configured with a custom scorer that converts predictions back to RM before computing RMSE, using the same fold split (`KFold(n_splits=5, shuffle=True, random_state=42)`) as the CV reported later in this section, so the two are directly comparable. `X_test` is not used anywhere in this tuning section.

In [3]:
def rm_scale_rmse(y_true_log, y_pred_log):
    y_true_rm = np.exp(y_true_log)
    y_pred_rm = np.exp(y_pred_log)
    return np.sqrt(np.mean((y_true_rm - y_pred_rm) ** 2))

rm_rmse_scorer = make_scorer(rm_scale_rmse, greater_is_better=False)
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "n_neighbors": [3, 5, 7, 10, 15, 20, 30],
    "weights": ["uniform", "distance"],
    "p": [1, 1.5, 2, 3],
}

grid_search = GridSearchCV(
    KNeighborsRegressor(),
    param_grid,
    scoring=rm_rmse_scorer,
    cv=cv_splitter,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV score (RM-scale RMSE): RM {-grid_search.best_score_:,.0f}")


Best params: {'n_neighbors': 3, 'p': 2, 'weights': 'distance'}
Best CV score (RM-scale RMSE): RM 208,347


## 5-fold Cross-Validation

Run on `X_train` only (`X_test` stays untouched), using the same fold split as the search above, so the CV score `GridSearchCV` selected on and the full metric breakdown here are directly comparable.

In [4]:
cv_results = cross_validate_model(
    KNeighborsRegressor(**grid_search.best_params_),
    X_train, y_train, n_splits=5,
)


5-fold CV (mean +/- std):
  RMSE:  RM 208,382 +/- 20,739  (59.3% of median price)
  MAE:   RM 97,263 +/- 4,161
  MAPE:  22.7% +/- 1.3%
  R2:    0.5887 +/- 0.0161


## Train Final Model

`GridSearchCV` was run with its default `refit=True`, so it already retrained a model on the full `X_train` using the best-found configuration (`grid_search.best_estimator_`). That model is reused directly here instead of manually reconstructing and refitting an identical one. `X_test` is not used in any hyperparameter decision - it is touched only once, below, for the final held-out evaluation.

In [5]:
final_params = grid_search.best_params_
model = grid_search.best_estimator_

print("Final params:", final_params)
print("Model ready (reused from GridSearchCV's refit=True).")


Final params: {'n_neighbors': 3, 'p': 2, 'weights': 'distance'}
Model ready (reused from GridSearchCV's refit=True).


## Evaluate
Metrics computed on both train and test sets - the gap between them is needed for Section 6.2's overfitting/underfitting analysis.

In [6]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")


Train RMSE:  RM 6,749  (1.9% of median price)
Train MAE:   RM 1,135
Train MAPE:  0.3%
Train R2:    0.9996

Test RMSE:  RM 190,051  (52.8% of median price)
Test MAE:   RM 95,110
Test MAPE:  20.4%
Test R2:    0.6718


## Sanity check: permutation importance vs EDA (Section 4.5.2)
KNN has no `coef_`/`feature_importances_` (predictions are based purely on distances to neighbouring points), so permutation importance is used instead: each feature is shuffled and the resulting drop in test R² is measured. Section 4.5.2 ranked Property Size, Bathroom, Parking Lot, and the Has_Gymnasium/Has_Swimming_Pool amenities as the strongest numerical correlates of price - this checks whether KNN's learned importances broadly agree.

In [7]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model, X_test, y_test, scoring="r2", n_repeats=10, random_state=42, n_jobs=-1,
)
importance_table = pd.Series(perm_result.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("Top 15 features by permutation importance (mean R2 drop):")
print(importance_table.head(15))


Top 15 features by permutation importance (mean R2 drop):
Property Size                     0.053433
PropertyType_Flat                 0.037643
Bathroom                          0.034282
State_Sabah                       0.030981
State_Unknown                     0.019410
PropertyType_Service_Residence    0.019121
PropertyType_Condominium          0.015502
State_Penang                      0.013661
State_Negeri_Sembilan             0.012581
Total Units                       0.012536
State_Sarawak                     0.012247
State_Putrajaya                   0.012046
Property Age                      0.010379
# of Floors                       0.010106
State_Kuala_Lumpur                0.009730
dtype: float64


## Save Trained Model
Saved for the Streamlit prototype (Section 8) to load directly, without retraining.

In [8]:
import joblib
model_path = os.path.join(MODEL_DIR, "knn_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")


Model saved to ..\models\knn_model.pkl
